# FIT4112 · LAB 4 · Notebook 04 (TỰ CHỌN)
## ARIMA: dự báo cố định 24 giờ từ cuối tập train

Notebook này **không thuộc bài nộp bắt buộc**. Khác Notebook 03 (rolling one-hour-ahead), ta **không** giả định có số đo mới trong 24 giờ sắp tới. So sánh ARIMA với baseline cố định là **giữ nguyên số đo cuối train cho cả 24 bước**, không lấy kết quả one-step của Notebook 03 để so trực tiếp.

ARIMA dùng bản đã forward-fill để fit, nhưng đánh giá **chỉ trên giờ có nhãn đo thật**.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab4.py").exists()), None)
assert ROOT is not None, "Mở notebook trong thư mục repo fit4112_lab_4."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from src.lab4 import error_metrics

path = ROOT / "data/processed/hourly_pm25.csv"
assert path.exists(), "Hãy chạy Notebook 01 trước."
hourly = pd.read_csv(path, parse_dates=["datetime"]).set_index("datetime")
cut = int(len(hourly) * 0.8)
train = hourly.iloc[:cut]
test = hourly.iloc[cut:cut+24].copy()
assert len(test) == 24
# 14 ngày cuối train để thử nghiệm nhanh; toàn bộ vẫn là dữ liệu trước test
history = train["pm25"].iloc[-24*14:].asfreq("h")
fitted = ARIMA(history, order=(1, 0, 0)).fit()
forecast = np.asarray(fitted.forecast(steps=24), dtype=float)
fixed_origin = np.full(24, float(history.iloc[-1]))
valid = test["observed_target"].astype(bool).to_numpy()
print("Số giờ có nhãn đo thật để đánh giá:", int(valid.sum()))
display(pd.DataFrame([
    {"method": "Last-observation (fixed origin)", **error_metrics(test["pm25"].to_numpy()[valid], fixed_origin[valid])},
    {"method": "ARIMA(1,0,0)", **error_metrics(test["pm25"].to_numpy()[valid], forecast[valid])}
]).round(3))
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(test.index, test["pm25"], label="Actual (includes ffill; score observed only)")
ax.plot(test.index, fixed_origin, label="Fixed-origin baseline")
ax.plot(test.index, forecast, label="ARIMA 24h")
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

**Tự suy nghĩ:** Vì sao MAE của ARIMA 24h fixed-origin không so trực tiếp với MAE RandomForest one-step rolling? ARIMA(1,0,0) có mô tả được chu kỳ 24h không? Có nên thử mô hình mùa vụ sau này?